In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
from transformers import AutoTokenizer, AutoModel

from Attention import attention

tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
model = AutoModel.from_pretrained("bert-base-chinese")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Position Wise Decoder

## Muti-Header Attention Block

In [3]:
def clones(module, N):
    """生成 N 个结构相同但参数独立的层。"""
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [4]:
class MultiHeadsAttention(nn.Module):
    """
        Multi-Head Attention Layer:
            batch_size  :
            d_model     :
            n_heads     : 头数
            seq_len     : 句子长度
    """
    def __init__(self, d_model, h, dropout=0.1):
        super(MultiHeadsAttention, self).__init__()
        assert d_model % h == 0 # 断言投影矩阵维度可以整除头数
        self.d_model = d_model
        self.n_heads = h
        self.d_k = d_model // h
        self.linears = clones(nn.Linear(d_model, d_model), 4)#QKVO四个线性投影变换
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]
        x, self.attn = attention(query, key, value, mask=mask, dropout=self.dropout)



## Encoder

## Code Test